In [1]:
!pip install langchain_core
!pip install langchain_groq
!pip install chromadb
!pip install transformers
!pip install huggingface-hub sentence-transformers wget langchain

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for wget: filename=wget-3.2-py3-none-any.whl size=9686 sha256=b009ab95353da5027f85945df4279a2a670db7e4e2eba9cce99609e9d025ca23
  Stored in directory: /Users/anirudh/Library/Caches/pip/wheels/9c/b9/25/66d2377ed05ab1424fa6b361f1088fc5ae065f96efad7202dc
Successfully built wget


In [2]:
!pip install langchain-community docarray

In [8]:
def warn (*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings("ignore")

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import DocArrayInMemorySearch
from langchain_huggingface import HuggingFaceEmbeddings

from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate,ChatPromptTemplate,MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.messages import HumanMessage,AIMessage,SystemMessage

import requests



In [9]:
llm = ChatGroq(
    model = "llama-3.3-70b-versatile",
    groq_api_key = "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",
    temperature = 0.5,
    max_tokens = 1024
)
filename = 'companyPolicies.txt'
url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/6JDbUb_L3egv_eOkouY71A.txt'
response = requests.get(url)
with open(filename ,"wb") as file:
    file.write(response.content)
    

In [10]:
#Load document
loader = TextLoader(filename)
documents = loader.load()
print(f" Loaded {len(documents)} documents")

 Loaded 1 documents


In [15]:
#Split document into chunks
splitter = CharacterTextSplitter(chunk_size = 1000, chunk_overlap = 20 ,separator = "\n")
chunks = splitter.split_documents(documents)
print(f" No of chunks are {len(chunks)}")

 No of chunks are 19


In [16]:
#Create embeddings
embeddings = HuggingFaceEmbeddings(
    model = "sentence-transformers/all-MiniLM-L6-V2"
)

In [18]:
vectorstore = DocArrayInMemorySearch.from_documents(chunks, embeddings)

In [22]:
#Create retriever
retriever = vectorstore.as_retriever(search_kwargs = {"k": 3})


In [20]:
# Basic RAG chain
def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])
    

In [21]:
#Simple RAG prompt
rag_template = """Answer the question based only on the following context:

{context}

Question: {question}

Answer:"""

rag_prompt = ChatPromptTemplate.from_template(rag_template)


In [27]:
# Create RAG chain
rag_chain = (
    {"context": retriever| format_docs ,"question": RunnablePassthrough()}
    | rag_prompt
    | llm
    |StrOutputParser()
)
query = """ Can you summarize the document fo me?"""
answer = rag_chain.invoke(query)
print(f" the query is {query}")
print(f" The answer is {answer}")

 the query is  Can you summarize the document fo me?
 The answer is The document outlines the organization's Discipline and Termination Policy, which aims to maintain a fair, respectful, and productive workplace. It emphasizes the importance of adherence to a Code of Conduct that values integrity, respect, and accountability. The policy applies to all personnel and outlines expectations for performance and conduct, as well as the disciplinary actions that may be taken if these expectations are not met. The goal is to address issues constructively and maintain a positive work environment.


In [31]:
# Creating RAG with custom prompts
custom_template = """You are a helpful AI assistant. Use the following context to answer the question.
If you don't know the answer, say "I don't have enough information to answer that."

Context:
{context}

Question: {question}

Helpful Answer:"""
custom_prompt = ChatPromptTemplate.from_template(custom_template)
custom_rag_chain =(
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    |custom_prompt
    |llm
    |StrOutputParser()
)
answer = custom_rag_chain.invoke("What is the main topic?.")
print(f" Answer is {answer}")

 Answer is The main topic appears to be the organization's policies, specifically their Code of Conduct, Recruitment Policy, and Health and Safety Policy, which outline the principles and guidelines for employee behavior, discipline, and termination, as well as the organization's commitment to safety, environmental responsibility, and ethical conduct.


In [34]:
# Conversational RAG with memory
conversational_template = """You are a helpful assistant. Use the context to answer questions.

Context:
{context}

Conversation history:
{chat_history}

Current question: {question}

Answer:"""
conversational_prompt = ChatPromptTemplate.from_template(conversational_template)
#Memory
chat_history = ChatMessageHistory()
def conversational_rag(query):
    """Rag with conversational memory """
    #Get relavant documents
    docs = retriever.invoke(query)
    context = format_docs(docs)
    #Format chat history
    history_text = "\n".join([f"{"User" if msg.type == "human" else "Assistant"}:{msg.content}"
    for msg in chat_history.messages])
    #Create prompt
    prompt_value = conversational_prompt.invoke({
        "context": context,
        "chat_history": history_text,
        "question":query
    })
    #Get response
    response = llm.invoke(prompt_value)
    #update history
    chat_history.add_user_message(query)
    chat_history.add_ai_message(response.content)
    return response.content
 #Have a conversation
print("\nQ1: What is this document about?")
answer1 = conversational_rag("What is this document about?")
print(f"A1: {answer1}")

print("\nQ2: Can you elaborate on that?")
answer2 = conversational_rag("Can you elaborate on that?")
print(f"A2: {answer2}")

print("\nQ3: What did I ask you first?")
answer3 = conversational_rag("What did I ask you first?")
print(f"A3: {answer3}")    
    
    


Q1: What is this document about?
A1: This document appears to be about the organization's policies, specifically the Code of Conduct and other related policies such as discipline and termination, recruitment, and overall workplace culture. It outlines the principles and ethical standards that guide the organization's behavior and decision-making.

Q2: Can you elaborate on that?
A2: This document outlines the organization's policies and guidelines for various aspects of the workplace. It covers topics such as the responsible use of mobile phones, internet, and email, as well as the recruitment process and the company's commitment to diversity and inclusion. The policies aim to promote a culture of transparency, security, and compliance with laws and regulations.

The document also emphasizes the importance of separating personal and company-related activities, such as keeping personal phone usage separate from company accounts and using company-provided internet and email services prim

In [38]:
# RAG with source retrieval
def rag_with_sources(query):
    """ Rag that returns output and source documents """
    source_docs = retriever.invoke(query)
    context = format_docs(source_docs)
    #Get answer
    prompt_value = rag_prompt.invoke({
        "context":context,
        "question":query
    })
    answer = llm.invoke(prompt_value)
    return {
        "answer":answer,
        "sources":source_docs
    }
result = rag_with_sources("What are the key points?")
print(f"Answer: {result['answer']}")
print(f"\nSource Documents:")
for i, doc in enumerate(result['sources'], 1):
    print(f"\nSource {i}:")
    print(f"Content: {doc.page_content[:200]}...")
    print(f"Metadata: {doc.metadata}")    
        
    

Answer: content='The key points of the Recruitment Policy are:\n\n1. Equal Opportunity: The company is an equal opportunity employer and does not discriminate on the basis of various protected statuses.\n2. Transparency: The company maintains transparency in its recruitment processes, including advertising job vacancies and providing clear job descriptions.\n3. Selection Criteria: The selection process is based on qualifications, experience, and skills necessary for the position, with objective interviews and assessments.\n4. Accountability: The company takes responsibility for its actions and decisions, follows relevant laws and regulations, and strives to continuously improve its practices.\n\nAdditionally, the company prioritizes diversity and inclusion, and expects all employees to uphold the principles of the Code of Conduct, which includes safety, environmental responsibility, and ethical conduct.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 148, 

In [41]:
#Interactive RAG chatbot
def interactive_rag_chatbot():
    """ Interactive chatbot with RAG and memory """
    history = ChatMessageHistory()
    while True:
        user_input = input("You :")
        if user_input.lower() in ['quit','exit','bye']:
            break
        docs = retriever.invoke(user_input)
        context = format_docs(docs)
        #Format history
        history_messages = "\n".join([
            f"{"User" if msg.type == "human" else " Assistant"}:{msg.content}"
        for msg in history.messages[-6:]])
        #Create prompt
        prompt_value = conversational_prompt.invoke({
            "context":context,
            "chat_history":history_messages,
            "question": user_input
        })
        #LLM response
        response = llm.invoke(prompt_value)
        #Adding data to chat history
        history.add_user_message(user_input)
        history.add_ai_message(response.content)
        print(f"Chatbot:{response.content}\n")
interactive_rag_chatbot()        

You : Hello


Chatbot:Hello! It's nice to meet you. I see we have some information about an organization's code of conduct and recruitment policy. Is there something specific you'd like to know or discuss about these topics? I'm here to help.



You : Summarize the document in 5 lines


Chatbot:The organization has a Discipline and Termination Policy to maintain a respectful workplace. 
The policy outlines expectations for employee conduct and performance. 
It emphasizes fairness, consistency, and respect in all interactions. 
Disciplinary actions will be taken when necessary to address issues constructively. 
The goal is to create a productive, ethical, and inclusive work environment for all personnel.



You : quit
